In [ ]:
import os
import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


start, end = '2016-01-01', '2025-05-31'
AETHALOMETER_FILE = os.path.join('Pasta', 'BC_ATOLL_hourly_2016_2025.csv')
AERONET_FILE = os.path.join('Pasta', '20160101_20250531_Lille_all.txt')
EDGAR_FILE = os.path.join('pasta', 'INTERPLAY_simplified_2025.csv')

def load_results(file_path):
    with open(file_path, 'rb') as f:
        return pickle.load(f)


results_1 = load_results(os.path.join('Pasta', 'results_2016_2024.pkl'))
results_2 = load_results(os.path.join('Pasta', 'results_2024_2025.pkl'))

BT_emi_sum = {}
BT_emi_sum.update(results_1.get('BT_emi_sum', {}))
BT_emi_sum.update(results_2.get('BT_emi_sum', {}))

BT_end_date_dt = pd.to_datetime(end) + pd.Timedelta(days=3, hours=23)
date_range = pd.date_range(start, BT_end_date_dt, freq='H')
df1 = pd.DataFrame(index=date_range, columns=['emi/BT'], dtype=float).reset_index().rename(columns={'index': 'DateTime'})

for traj_name, emission in BT_emi_sum.items():
    try:
        date_str = ''.join(filter(str.isdigit, os.path.basename(traj_name)))[-8:]
        dt_key = pd.to_datetime(date_str, format='%y%m%d%H')
        dt_key = dt_key.floor('H')
        if dt_key in df1['DateTime'].values:
            df1.loc[df1['DateTime'] == dt_key, 'emi/BT'] = emission
    except (ValueError, IndexError):
        continue

try:
    aethalometer_df = pd.read_csv(AETHALOMETER_FILE, sep='; ', engine='python', skiprows=1)
    aeronet_df = pd.read_csv(AERONET_FILE, sep=',', engine='python', skiprows=6)
except FileNotFoundError as e:
    raise SystemExit(f"CRITICAL ERROR: File not found: {e}")

aet_date = pd.to_datetime(aethalometer_df['Time (UTC, start of average)'], format='%d-%b-%Y %H:%M:%S')
df2 = pd.DataFrame({
    'DateTime': aet_date,
    'BC6': aethalometer_df['BC6 (ug m-3)'],
    'BB': aethalometer_df['BB']
})
df2['DateTime'] = df2['DateTime'].dt.floor('H')

aer_date = pd.to_datetime(aeronet_df['Date(dd:mm:yyyy)'] + ' ' + aeronet_df['Time(hh:mm:ss)'], format='%d:%m:%Y %H:%M:%S')

aod_440 = aeronet_df['AOD_Extinction-Fine[440nm]']
aod_675 = aeronet_df['AOD_Extinction-Fine[675nm]']
ssa_440 = aeronet_df['Single_Scattering_Albedo[440nm]']
ssa_675 = aeronet_df['Single_Scattering_Albedo[675nm]']

aaod_440 = aod_440 * (1 - ssa_440)
aaod_675 = aod_675 * (1 - ssa_675)
aae = - (np.log(aaod_440 / aaod_675) / np.log(440 / 675))

saod_440 = aod_440 * ssa_440
saod_675 = aod_675 * ssa_675
sae = - (np.log(saod_440 / saod_675) / np.log(440 / 675))

df3 = pd.DataFrame({
    'DateTime': aer_date,
    'aae_fine': aae,
    'sae_fine': sae,
    'aod_fine_440': aod_440
})
df3['DateTime'] = df3['DateTime'].dt.floor('H')

df1['DateTime'] = df1['DateTime'].dt.floor('H')
general_df = pd.merge(df1, df2, on='DateTime', how='left')
general_df = pd.merge(general_df, df3, on='DateTime', how='left')

BC = pd.to_numeric(general_df['BC6'], errors='coerce')
BB = pd.to_numeric(general_df['BB'], errors='coerce')
general_df['BCwb'] = BC * BB / 100
general_df.set_index('DateTime', inplace=True)

df_edgar = pd.read_csv(EDGAR_FILE, header=[0, 1], sep=';')
df_edgar.columns = pd.MultiIndex.from_tuples([(a.strip(), b.strip()) for a, b in df_edgar.columns])

time_col = ('INTERPLAY-EDGAR', 'Arrival time (UTC)')
col_bc = ('file created on 10-Jul-25 08:29', 'integrated BC (Gg yr-1)')

df_edgar[time_col] = pd.to_datetime(df_edgar[time_col], errors='coerce')
df_edgar[col_bc] = pd.to_numeric(df_edgar[col_bc], errors='coerce')

A = 122.98e6  # m²
seconds_per_year = 3600 * 24 * 365
new_col = (col_bc[0], col_bc[1] + ' (kg/m²/s)')
df_edgar[new_col] = (df_edgar[col_bc] * 1e6) / A / seconds_per_year

df_edgar_filtered = df_edgar[df_edgar[time_col].notna() & df_edgar[new_col].notna()]
df_edgar_filtered = df_edgar_filtered.set_index(time_col)

emi_bt = general_df['emi/BT'].fillna(0)
emi_edgar = df_edgar_filtered[new_col]


emi_bt_by_year = {year: data for year, data in emi_bt.groupby(emi_bt.index.year)}
emi_edgar_by_year = {year: data for year, data in emi_edgar.groupby(emi_edgar.index.year)}


for year in range(2016, 2026):
    bt_data = emi_bt_by_year.get(year, pd.Series(dtype=float))
    edgar_data = emi_edgar_by_year.get(year, pd.Series(dtype=float))


    bt_data = bt_data.groupby(bt_data.index).sum()
    edgar_data = edgar_data.groupby(edgar_data.index).sum()

   
    year_index = pd.date_range(f'{year}-01-01', f'{year}-12-31 23:00', freq='H')
    df_year = pd.DataFrame(index=year_index)
    df_year['Forest Fires (GFAS)'] = bt_data
    df_year['Anthropogenic (EDGAR)'] = edgar_data

   
    df_year.dropna(how='all', inplace=True)
    if df_year.empty:
        continue

  
    total = df_year['Forest Fires (GFAS)'] + df_year['Anthropogenic (EDGAR)']
    total[total == 0] = np.nan
    df_year['Forest Fires (%)'] = (df_year['Forest Fires (GFAS)'] / total) * 100
    df_year['Anthropogenic (%)'] = (df_year['Anthropogenic (EDGAR)'] / total) * 100

    df_year.dropna(subset=['Forest Fires (%)', 'Anthropogenic (%)'], inplace=True)


    plt.figure(figsize=(14, 5), dpi=150)
    plt.stackplot(df_year.index,
                  df_year['Forest Fires (%)'],
                  df_year['Anthropogenic (%)'],
                  labels=['Forest Fires (GFAS)', 'Anthropogenic (EDGAR)'],
                  colors=['indianred', 'steelblue'],
                  alpha=0.8)

    plt.title(f'Percentual das Emissões Anuais de BC - {year}', fontsize=14, weight='bold')
    plt.ylabel('Emissões (%)', fontsize=12)
    plt.xlabel('Data')
    plt.legend(loc='upper right')
    plt.ylim(0, 100)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()